# Tổng quan về Logistic Regression

Chào mừng bạn đến với notebook về **Logistic Regression**! Đây là một thuật toán **học có giám sát** dùng cho bài toán **phân loại nhị phân** (dự đoán một trong hai lớp). Trong notebook này, chúng ta sẽ cùng nhau tìm hiểu từ nền tảng lý thuyết (hàm Sigmoid, Logit, Log Loss), cách cài đặt với thư viện scikit-learn, tầm quan trọng của việc chuẩn hóa dữ liệu, và cuối cùng là các chỉ số đánh giá mô hình. Mục tiêu là giúp bạn hiểu rõ bản chất và tự tin áp dụng Logistic Regression vào các bài toán thực tế.

## Dữ liệu: Heart Failure Prediction Dataset

Chúng ta sẽ sử dụng bộ dữ liệu **Heart Failure Prediction Dataset** (nguồn: Kaggle) để dự đoán khả năng mắc bệnh tim của một người dựa trên các đặc trưng lâm sàng.

- **Kích thước:** 918 dòng, 12 cột (11 đặc trưng + 1 nhãn).
- **Biến mục tiêu:** `HeartDisease` (0 = không bệnh, 1 = có bệnh tim mạch).
- **Đặc trưng chính:** `Age`, `Sex`, `ChestPainType`, `RestingBP`, `Cholesterol`, `FastingBS`, `RestingECG`, `MaxHR`, `ExerciseAngina`, `Oldpeak`, `ST_Slope`.
- **Tiền xử lý:** Các giá trị 0 bất thường (ví dụ `Cholesterol = 0`) được coi là giá trị khuyết và đã được xử lý bằng median. Dữ liệu đã được chuẩn hóa bằng `StandardScaler`.

### 1. Load Dataset & Inspection

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

relative_data_path = Path("notebookforge/datasets/heart.csv")
data_candidates = [
    Path.cwd() / relative_data_path,
    Path.cwd() / "datasets" / relative_data_path.name,
    Path.cwd().parent / "datasets" / relative_data_path.name,
]
data_path = next(
    (path.resolve() for path in data_candidates if path.is_file()),
    Path.cwd() / relative_data_path,
)

# Kiểm tra sự tồn tại của Dataset File
if not data_path.is_file():
    raise FileNotFoundError(
        f"❌ KHÔNG TÌM THẤY DATASET: 'Heart Failure Prediction' tại đường dẫn '{os.path.abspath(data_path)}'.\n"
        f"👉 Vui lòng đảm bảo bạn đã copy file CSV vào đúng thư mục 'notebookforge/datasets/'!"
    )

# Load Heart Failure Prediction Dataset
df = pd.read_csv(data_path)

print(f"Dataset Successfully Loaded! Shape: {df.shape}")
df.head()

### 2. EDA & Handling Missing/Outlier Values

In [ ]:
# 1. Loại bỏ dòng vô lý RestingBP = 0
df = df.copy()
df['RestingBP'] = df['RestingBP'].replace(0, np.nan)

# 2. Chuyển Cholesterol = 0 thành NaN để Impute (Dữ liệu khuyết ngầm y khoa)
df['Cholesterol'] = df['Cholesterol'].replace(0, np.nan)

print("Kiểm tra giá trị Null trước khi chia dữ liệu:")
print(df.isnull().sum())

### 3. Feature Encoding, Scaling & Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# 1. One-Hot Encoding cho các biến Categorical (Sex, ChestPainType, RestECG, ExerciseAngina, ST_Slope)
X = pd.get_dummies(df.drop(columns=['HeartDisease']), drop_first=True)
y = df['HeartDisease']

# 2. Chia Train/Test (Stratify theo nhãn)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Fit imputer/scaler chỉ trên train để tránh data leakage.
imputer = SimpleImputer(strategy='median')
X_train = pd.DataFrame(
    imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test = pd.DataFrame(
    imputer.transform(X_test), columns=X_test.columns, index=X_test.index
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ X_train shape: {X_train_scaled.shape}, X_test shape: {X_test_scaled.shape}")

## Module m1: Nền tảng lý thuyết: Sigmoid, Logit và Log Loss

**Mục tiêu:** Hiểu được bản chất của Logistic Regression: cách biến đổi tổ hợp tuyến tính thành xác suất thông qua hàm sigmoid, ý nghĩa của logit (log-odds), và cách hàm mất mát log loss đo lường sai số dự đoán.

### Hàm Sigmoid

Hàm sigmoid là một hàm toán học có dạng chữ "S", được sử dụng để ánh xạ một giá trị thực bất kỳ về khoảng (0, 1), giúp chúng ta diễn giải kết quả như một xác suất. Công thức của hàm sigmoid là:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Trong đó, $z$ là sự kết hợp tuyến tính giữa các đặc trưng (features) và trọng số (weights):

$$z = w_0 + w_1 x_1 + w_2 x_2 + ... + w_n x_n = \mathbf{w}^T \mathbf{x} + b$$

### Logit (Log-Odds)

Logistic Regression không dự đoán trực tiếp xác suất mà dự đoán **log-odds** (logit) của biến mục tiêu. Logit là logarit tự nhiên của tỷ số odds (tỷ lệ giữa xác suất xảy ra sự kiện và xác suất không xảy ra). Mô hình này được gọi là "tuyến tính" vì nó tạo ra một mối quan hệ tuyến tính giữa các đặc trưng và log-odds.

### Log Loss

Log Loss (hay Binary Cross-Entropy) là hàm mất mát được sử dụng để đo lường mức độ sai lệch giữa xác suất dự đoán và nhãn thực tế. Nó phạt nặng các dự đoán sai một cách tự tin, giúp mô hình học được các xác suất chính xác hơn.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Demo: Vẽ đồ thị hàm sigmoid
z = np.linspace(-10, 10, 100)
sigmoid = 1 / (1 + np.exp(-z))

plt.figure(figsize=(6, 4))
plt.plot(z, sigmoid, label='Sigmoid function')
plt.axhline(0.5, color='gray', linestyle='--', linewidth=0.8)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.8)
plt.xlabel('z')
plt.ylabel('σ(z)')
plt.title('Đồ thị hàm Sigmoid')
plt.legend()
plt.grid(True)
plt.show()

# Hàm sigmoid cho một giá trị cụ thể
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

print(f"Sigmoid(0) = {sigmoid(0):.4f}")
print(f"Sigmoid(1.5) = {sigmoid(1.5):.4f}")

### Bài tập m1_ex1: Tính xác suất bằng hàm sigmoid

**exercise_id:** m1_ex1

**Đề bài:** Cho z = 1.5, hãy tính xác suất dự đoán bằng công thức sigmoid. Viết code Python để tính và in kết quả.

**Gợi ý:** Sử dụng hàm `sigmoid(z)` đã định nghĩa ở trên.

In [ ]:
# TODO: Tính sigmoid cho z = 1.5 và lưu vào biến m1_ex1_result
z_value = 1.5
m1_ex1_result = None  # Thay None bằng lời giải của bạn

# In kết quả
if m1_ex1_result is not None:
    print(f"Sigmoid({z_value}) = {m1_ex1_result:.6f}")

## Module m2: Cài đặt với scikit-learn: LogisticRegression

**Mục tiêu:** Nắm được cách sử dụng class `LogisticRegression` trong scikit-learn, các phương thức `fit`/`predict`/`predict_proba`, và các hyperparameter quan trọng như `penalty`, `solver`, `class_weight`.

### LogisticRegression trong scikit-learn

`LogisticRegression` là một class mạnh mẽ trong thư viện scikit-learn, cho phép chúng ta huấn luyện mô hình logistic regression một cách dễ dàng. Các tham số quan trọng:

- **`penalty`**: Kỹ thuật regularization (L1, L2) giúp tránh overfitting.
- **`solver`**: Thuật toán tối ưu (ví dụ: 'lbfgs', 'liblinear').
- **`class_weight`**: Xử lý mất cân bằng lớp (ví dụ: 'balanced').

Pipeline chuẩn thường được sử dụng:

```python
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(
        penalty='l2',
        C=1.0,
        solver='lbfgs',
        max_iter=1000,
        class_weight='balanced',
        random_state=42
    ))
])
```

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Demo: Huấn luyện mô hình LogisticRegression
# Giả sử X_train, y_train đã được tạo bởi Dataset Injector

# Tạo pipeline với StandardScaler và LogisticRegression
model = LogisticRegression(
    penalty='l2',
    C=1.0,
    solver='lbfgs',
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)

# Huấn luyện mô hình
model.fit(X_train, y_train)

# Dự đoán trên tập test
y_pred = model.predict(X_test)

# Dự đoán xác suất
y_pred_proba = model.predict_proba(X_test)

print(f"Kích thước X_train: {X_train.shape}")
print(f"Kích thước X_test: {X_test.shape}")
print(f"Hệ số (coef_): {model.coef_}")
print(f"Xác suất dự đoán cho 5 mẫu đầu tiên:")
print(y_pred_proba[:5])

### Bài tập m2_ex1: Huấn luyện mô hình LogisticRegression

**exercise_id:** m2_ex1

**Đề bài:** Sử dụng bộ dữ liệu có sẵn (X_train, y_train), huấn luyện mô hình LogisticRegression với `penalty='l2'`, `solver='lbfgs'`, `class_weight='balanced'`. Sau đó dự đoán trên X_test và in ra xác suất dự đoán cho 5 mẫu đầu tiên.

**Gợi ý:** Sử dụng class `LogisticRegression` từ scikit-learn và phương thức `fit`, `predict_proba`.

In [ ]:
# TODO: Huấn luyện mô hình và lưu vào biến m2_ex1_model
m2_ex1_model = None  # Thay None bằng lời giải của bạn

# Dự đoán xác suất cho 5 mẫu đầu tiên của X_test
if m2_ex1_model is not None:
    m2_ex1_proba = m2_ex1_model.predict_proba(X_test)
    print("Xác suất dự đoán cho 5 mẫu đầu tiên:")
    print(m2_ex1_proba[:5])

## Module m3: Tiền xử lý dữ liệu: StandardScaler

**Mục tiêu:** Hiểu vì sao cần chuẩn hóa dữ liệu (StandardScaler) trước khi huấn luyện Logistic Regression, và cách nó ảnh hưởng đến tốc độ hội tụ và hiệu quả của mô hình.

### StandardScaler

`StandardScaler` là một kỹ thuật chuẩn hóa dữ liệu, giúp biến đổi các đặc trưng về cùng một tỷ lệ bằng cách trừ đi giá trị trung bình và chia cho độ lệch chuẩn. Công thức:

$$x_{scaled} = \frac{x - \mu}{\sigma}$$

Trong đó $\mu$ là giá trị trung bình và $\sigma$ là độ lệch chuẩn của đặc trưng.

**Tại sao cần chuẩn hóa?**

- **Tăng tốc độ hội tụ:** Khi các đặc trưng có tỷ lệ khác nhau, thuật toán tối ưu (như gradient descent) sẽ hội tụ chậm hơn.
- **Cải thiện hiệu quả:** Một số thuật toán nhạy cảm với tỷ lệ của đặc trưng, chuẩn hóa giúp mô hình hoạt động tốt hơn.
- **Tránh nhiễu:** Các đặc trưng có giá trị lớn có thể lấn át các đặc trưng có giá trị nhỏ.

In [ ]:
from sklearn.preprocessing import StandardScaler

# Demo: Chuẩn hóa dữ liệu với StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# So sánh trước và sau khi chuẩn hóa
print("Trước khi chuẩn hóa:")
print(f"  - Mean của cột đầu tiên: {X_train.iloc[:, 0].mean():.4f}")
print(f"  - Std của cột đầu tiên: {X_train.iloc[:, 0].std():.4f}")

print("\nSau khi chuẩn hóa:")
print(f"  - Mean của cột đầu tiên: {X_train_scaled[:, 0].mean():.4f}")
print(f"  - Std của cột đầu tiên: {X_train_scaled[:, 0].std():.4f}")

# Vẽ biểu đồ so sánh phân phối trước và sau khi chuẩn hóa
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].hist(X_train.iloc[:, 0], bins=20, alpha=0.7, color='blue')
axes[0].set_title('Trước khi chuẩn hóa')
axes[0].set_xlabel('Giá trị')

axes[1].hist(X_train_scaled[:, 0], bins=20, alpha=0.7, color='green')
axes[1].set_title('Sau khi chuẩn hóa')
axes[1].set_xlabel('Giá trị')

plt.tight_layout()
plt.show()

## Module m4: Đánh giá mô hình: Confusion Matrix, Precision, Recall, F1-Score, ROC-AUC

**Mục tiêu:** Biết cách sử dụng các chỉ số đánh giá phân loại: Confusion Matrix, Precision, Recall, F1-Score, và ROC-AUC để đánh giá chất lượng mô hình Logistic Regression.

### Confusion Matrix

Ma trận nhầm lẫn (Confusion Matrix) là bảng thống kê số lượng dự đoán đúng và sai của mô hình phân loại nhị phân:

| | Dự đoán Negative ($y=0$) | Dự đoán Positive ($y=1$) |
| :--- | :--- | :--- |
| **Thực tế Negative ($y=0$)** | True Negative (**TN**) | False Positive (**FP**) *(Lỗi Type I)* |
| **Thực tế Positive ($y=1$)** | False Negative (**FN**) *(Lỗi Type II)* | True Positive (**TP**) |

### Precision, Recall, F1-Score

- **Precision**: Tỷ lệ dự đoán positive đúng trong tổng số dự đoán positive: $\text{Precision} = \frac{TP}{TP + FP}$
- **Recall**: Tỷ lệ dự đoán đúng các mẫu positive thực tế: $\text{Recall} = \frac{TP}{TP + FN}$
- **F1-Score**: Trung bình điều hòa (Harmonic Mean) giữa Precision và Recall, giúp đánh giá sự cân bằng giữa hai chỉ số này:

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}} = \frac{2 \cdot \text{TP}}{2 \cdot \text{TP} + \text{FP} + \text{FN}}$$

### ROC-AUC

- **ROC Curve (Receiver Operating Characteristic)**: Đồ thị biểu diễn mối tương quan giữa **True Positive Rate (Recall)** trên trục tung và **False Positive Rate** ($\text{FPR} = \frac{\text{FP}}{\text{TN} + \text{FP}}$) trên trục hoành khi thay đổi ngưỡng quyết định từ $1.0$ về $0.0$.
- **ROC-AUC Score**: Diện tích nằm dưới đường cong ROC ($0.5 \le \text{AUC} \le 1.0$).
  - $\text{AUC} = 0.5$: Mô hình dự đoán ngẫu nhiên (Random Guessing).
  - $\text{AUC} = 1.0$: Mô hình phân tách hai lớp hoàn hảo.

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

# Demo: Đánh giá mô hình với các chỉ số
# Sử dụng mô hình đã huấn luyện ở Module m2

# Tính Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(cm)

# Tính các chỉ số
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba[:, 1])

print(f"\nPrecision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")

# Vẽ Confusion Matrix bằng matplotlib
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set(xticks=np.arange(cm.shape[1]),
       yticks=np.arange(cm.shape[0]),
       xticklabels=['Negative', 'Positive'],
       yticklabels=['Negative', 'Positive'],
       ylabel='Thực tế',
       xlabel='Dự đoán')

# Thêm số vào các ô
thresh = cm.max() / 2.
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], 'd'),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")

plt.title('Confusion Matrix')
plt.show()

# Vẽ ROC Curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba[:, 1])
plt.figure(figsize=(6, 4))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.grid(True)
plt.show()

### Bài tập m4_ex1: Tính các chỉ số đánh giá

**exercise_id:** m4_ex1

**Đề bài:** Sau khi huấn luyện mô hình, tính Confusion Matrix, Precision, Recall, F1-Score và ROC-AUC trên tập test. In ra tất cả các giá trị.

**Gợi ý:** Sử dụng các hàm `confusion_matrix`, `precision_score`, `recall_score`, `f1_score`, `roc_auc_score` từ `sklearn.metrics`.

In [ ]:
# TODO: Tính các chỉ số đánh giá và lưu vào biến m4_ex1_roc_auc
m4_ex1_roc_auc = None  # Thay None bằng lời giải của bạn

# In các chỉ số
if m4_ex1_roc_auc is not None:
    print(f"ROC-AUC: {m4_ex1_roc_auc:.4f}")

## Khối 4: Kiểm tra kết quả

Phần này chứa các bài kiểm tra tự động để xác nhận bạn đã hoàn thành đúng các bài tập.

In [ ]:
try:
    # Kiểm tra m1_ex1
    if m1_ex1_result is not None:
        assert abs(m1_ex1_result - 0.8175744761936437) < 1e-6
        print("Bài tập m1_ex1: Đúng!")
    else:
        print("Bài tập m1_ex1: Hãy hoàn thành bài tập trước khi kiểm tra.")
except (AssertionError, TypeError, ValueError, NameError, NotImplementedError) as exercise_error:
    print(f'Bài tập chưa hoàn thành: {exercise_error}')

In [ ]:
try:
    # Kiểm tra m2_ex1
    if m2_ex1_model is not None:
        assert hasattr(m2_ex1_model, 'coef_') and m2_ex1_model.coef_.shape[1] == X_train.shape[1]
        print("Bài tập m2_ex1: Đúng!")
    else:
        print("Bài tập m2_ex1: Hãy hoàn thành bài tập trước khi kiểm tra.")
except (AssertionError, TypeError, ValueError, NameError, NotImplementedError) as exercise_error:
    print(f'Bài tập chưa hoàn thành: {exercise_error}')

In [ ]:
try:
    # Kiểm tra m4_ex1
    if m4_ex1_roc_auc is not None:
        assert m4_ex1_roc_auc > 0.5
        print("Bài tập m4_ex1: Đúng!")
    else:
        print("Bài tập m4_ex1: Hãy hoàn thành bài tập trước khi kiểm tra.")
except (AssertionError, TypeError, ValueError, NameError, NotImplementedError) as exercise_error:
    print(f'Bài tập chưa hoàn thành: {exercise_error}')